# Amazon Reviews 2023 — Bronze Layer → MinIO



In [2]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 483.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 400.2 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 215.7 kB/s eta 0:00:00a 0:00:01


In [1]:
import os
import gzip
import json
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
from io import BytesIO

CATEGORY         = "Electronics"
MINIO_ENDPOINT   = "http://minio:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
MINIO_BUCKET     = "datalake"
BATCH_SIZE       = 200_000

LOCAL_DIR        = f"/mnt/data/amazon2023_raw/{CATEGORY}"
os.makedirs(LOCAL_DIR, exist_ok=True)

print(f"[INFO] Thu muc lam viec local: {LOCAL_DIR}")

[INFO] Thu muc lam viec local: /mnt/data/amazon2023_raw/Electronics


In [3]:
print("[INFO] Bat dau tai du lieu...")

!wget -c https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/{CATEGORY}.jsonl.gz -O {LOCAL_DIR}/reviews.jsonl.gz
!wget -c https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_{CATEGORY}.jsonl.gz -O {LOCAL_DIR}/meta.jsonl.gz

print("[DONE] Download hoan tat!")

[INFO] Bat dau tai du lieu...
--2026-03-29 06:33:24--  https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5, 137.110.161.5, 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.

--2026-03-29 06:33:25--  https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Electronics.jsonl.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5, 137.110.161.5, 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 206 Partial Content
Length: 1312900427 (1.2G), 1279363005 (1.2G) remaining [application/gzip]
Saving to: ‘/mnt/data/amazon2023_raw/Electronics/meta.jsonl.gz’



In [5]:
# ============================================================
# Setup MinIO Client
# ============================================================
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
)

def write_to_minio(batch_rows, schema, s3_key):
    table = pa.Table.from_pylist(batch_rows, schema=schema)
    buf = BytesIO()
    pq.write_table(table, buf, compression="snappy")
    buf.seek(0)
    
    s3.put_object(Bucket=MINIO_BUCKET, Key=s3_key, Body=buf.getvalue())
    print(f"  [OK] {s3_key} | {len(batch_rows):,} rows")

In [6]:
# ============================================================
# Stream reviews.jsonl.gz -> MinIO
# ============================================================
REVIEW_SCHEMA = pa.schema([
    ("user_id",           pa.string()),
    ("asin",              pa.string()),
    ("parent_asin",       pa.string()),
    ("rating",            pa.float64()),
    ("title",             pa.string()),
    ("text",              pa.string()),
    ("timestamp",         pa.int64()),
    ("helpful_vote",      pa.int32()),
    ("verified_purchase", pa.bool_()),
])

def parse_review(rec):
    uid = str(rec.get("user_id") or "").strip()
    if not uid:
        return None
    return {
        "user_id":           uid,
        "asin":              str(rec.get("asin") or "").strip() or None,
        "parent_asin":       str(rec.get("parent_asin") or "").strip() or None,
        "rating":            float(rec["rating"]) if rec.get("rating") is not None else None,
        "title":             str(rec.get("title") or "").strip() or None,
        "text":              str(rec.get("text") or "").strip() or None,
        "timestamp":         int(rec["timestamp"]) if rec.get("timestamp") is not None else None,
        "helpful_vote":      int(rec.get("helpful_vote") or 0),
        "verified_purchase": bool(rec["verified_purchase"]) if rec.get("verified_purchase") is not None else None,
    }

print("\n[INFO] Dang xu ly reviews...")
batch, batch_idx, total = [], 0, 0

with gzip.open(f"{LOCAL_DIR}/reviews.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
            
        row = parse_review(rec)
        if row is None:
            continue
            
        batch.append(row)
        if len(batch) >= BATCH_SIZE:
            key = f"bronze/reviews/{CATEGORY}/batch_{batch_idx:05d}.parquet"
            write_to_minio(batch, REVIEW_SCHEMA, key)
            total += len(batch)
            batch_idx += 1
            batch = []

# Ghi not so luong data con lai vao MinIO
if batch:
    key = f"bronze/reviews/{CATEGORY}/batch_{batch_idx:05d}.parquet"
    write_to_minio(batch, REVIEW_SCHEMA, key)
    total += len(batch)

print(f"[DONE] Xy ly reviews hoan tat! Tong {total:,} rows")


[INFO] Dang xu ly reviews...
  [OK] bronze/reviews/Electronics/batch_00000.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00001.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00002.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00003.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00004.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00005.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00006.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00007.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00008.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00009.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00010.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00011.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00012.parquet | 200,000 rows
  [OK] bronze/reviews/Electronics/batch_00013.parquet | 200,000 rows
  [O

In [7]:
## ============================================================
# Stream meta.jsonl.gz -> MinIO
# ============================================================
META_SCHEMA = pa.schema([
    ("parent_asin",    pa.string()),
    ("title",          pa.string()),
    ("main_category",  pa.string()),
    ("average_rating", pa.float64()),
    ("rating_number",  pa.int64()),
    ("price_raw",      pa.string()),
])

def parse_meta(rec):
    return {
        "parent_asin":    str(rec.get("parent_asin") or "").strip() or None,
        "title":          str(rec.get("title") or "").strip() or None,
        "main_category":  str(rec.get("main_category") or "").strip() or None,
        "average_rating": float(rec["average_rating"]) if rec.get("average_rating") is not None else None,
        "rating_number":  int(rec["rating_number"]) if rec.get("rating_number") is not None else None,
        "price_raw":      str(rec.get("price") or "").strip() or None,
    }

print("\n[INFO] Dang xu ly metadata...")
meta_rows = []

with gzip.open(f"{LOCAL_DIR}/meta.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
            
        row = parse_meta(rec)
        if row:
            meta_rows.append(row)

# Metadata co dung luong nho hon nen chi can ghi 1 file duy nhat
write_to_minio(meta_rows, META_SCHEMA, f"bronze/metadata/{CATEGORY}/meta.parquet")

print(f"[DONE] Xu ly metadata hoan tat! Tong {len(meta_rows):,} rows")


[INFO] Dang xu ly metadata...
  [OK] bronze/metadata/Electronics/meta.parquet | 1,610,012 rows
[DONE] Xu ly metadata hoan tat! Tong 1,610,012 rows
